# OpenAI Evals 入门

**注意：OpenAI 现在提供了带有 API 的托管 evals 产品！我们建议你改用它。
请参阅 [Evals](https://platform.openai.com/docs/guides/evals)**

[OpenAI Evals](https://github.com/openai/evals/tree/main) 框架包括
1. 一个用于评估 LLM（大语言模型）或基于 LLM 构建的系统的框架。
2. 一个开源的高难度 eval 注册表

本 notebook 将涵盖：
* 评估及 [OpenAI Evals](https://github.com/openai/evals/tree/main) 库简介
* 构建 Eval
* 运行 Eval

#### 什么是评估/`evals`？

评估是对 LLM 应用产生的输出进行验证和测试的过程。拥有完善的评估（"evals"）意味着应用会更稳定、更可靠，能够应对代码和模型的变化。eval 是用于衡量 LLM 或 LLM 系统输出质量的任务。给定一个输入提示词，会生成一个输出。我们用一组理想答案来评估该输出，从而得出 LLM 系统的质量。

#### 评估的重要性

如果你基于 `GPT-4` 等基础模型进行构建，创建高质量的 evals 是你能做的最有影响力的事情之一。开发 AI 解决方案是一个迭代的设计过程。[如果没有 evals，要了解](https://youtu.be/XGJNo8TpuVA?feature=shared&t=1089)不同的模型版本和提示词会如何影响你的用例，会非常困难且耗时。

随着 OpenAI 的[持续模型升级](https://platform.openai.com/docs/models/continuous-model-upgrades)，evals 让你能够以标准化的方式高效测试模型在你的用例上的表现。开发一套根据你的目标定制的 evals，将帮助你快速有效地了解新模型在你的用例上可能的表现。你还可以将 evals 纳入 CI/CD 流水线，确保在部署前达到预期的准确率。

#### eval 的类型

评估/评分补全结果主要有两种方式：用代码编写一些验证逻辑，
或者使用模型本身来检查答案。我们将通过一些示例分别介绍。

**编写检查答案的逻辑**

最简单、最常见的 eval 类型包含一个输入和一个理想的响应或答案。例如，
我们可以有一个 eval 样本，其输入是“奥巴马第一次当选总统是在哪一年？”，
理想答案是“2008”。我们将输入提供给模型并获得补全结果。如果模型
回答“2008”，就会被评为正确。我们可以编写字符串匹配来检查补全结果是否包含“2008”这个短语。如果包含，我们就认为它是正确的。

再考虑另一个 eval，其输入要求生成有效的 JSON：我们可以编写一些代码，
尝试将补全结果解析为 JSON，如果能够解析，
就认为补全结果是正确的。

**模型评分：一个两阶段过程，模型先回答问题，然后我们让
一个模型查看该响应，检查它是否正确。**

考虑一个要求模型写一个有趣笑话的输入。模型随后生成一个
补全结果。然后我们为模型创建一个包含该补全结果的新输入，让它回答：“下面这个
笑话好笑吗？先逐步推理，然后回答是或否”。最后，
如果新的模型补全结果以“是”结尾，我们就认为原始补全结果是正确的。

模型评分在使用 `GPT-4` 这类最新、最强大的模型，并让模型在做出判断前先进行推理时效果最好。
模型评分会有一定的错误率，因此在大规模运行 evals 之前，
用人工评估来验证其表现非常重要。为了获得最佳结果，
使用与生成补全结果的模型不同的模型来评分是合理的，例如使用 `GPT-4` 来
为 `GPT-3.5` 的答案评分。

#### OpenAI Eval 模板

在使用 evals 的过程中，我们发现了几种能够适配许多不同基准的“模板”。我们已在 OpenAI Evals 库中实现了这些模板，以简化新 evals 的开发。例如，我们定义了 2 种开箱即用的 eval 模板：

* **基础 Eval 模板**：包含将输出与 ideal_answers 进行比较的确定性函数。当期望的模型响应变化很小时，例如回答多项选择题或答案直截了当的简单问题，我们发现以下模板很有用。

* **模型评分模板**：包含由 LLM 将输出与 ideal_answers 进行比较并尝试判断准确性的函数。当期望的模型响应可能有很大变化时，例如回答开放式问题，我们发现使用模型为自身评分是一种可行的自动化评估策略。


### 环境设置

首先，前往 [github.com/openai/evals](https://github.com/openai/evals)，使用 `git clone git@github.com:openai/evals.git` 克隆仓库，并按照[设置说明](https://github.com/openai/evals)进行操作。 

为了在本 notebook 后面运行 evals，你需要设置并指定你的 OpenAI API 密钥。获取 API 密钥后，请通过 `OPENAI_API_KEY` 环境变量进行指定。 

运行 evals 时，请注意使用 API 所产生的费用。

In [1]:
from openai import OpenAI
import pandas as pd

client = OpenAI()

## 为 OpenAI Evals 框架构建评估

从本质上讲，eval 就是一个数据集加上一个在 YAML 文件中定义的 eval 类。要开始创建 eval，我们需要

1. `jsonl` 格式的测试数据集。
2. 要使用的 eval 模板

### 创建 eval 数据集
我们来为一个用例创建数据集，该用例评估模型生成语法正确的 SQL 的能力。在这个用例中，我们有一系列与汽车制造相关的表

首先，我们需要创建一个要评估的系统提示词。我们会传入给模型的指令以及表结构概览：
```
"TASK: Answer the following question with syntactically correct SQLite SQL. The SQL should be correct and be in context of the previous question-answer pairs.\nTable car_makers, columns = [*,Id,Maker,FullName,Country]\nTable car_names, columns = [*,MakeId,Model,Make]\nTable cars_data, columns = [*,Id,MPG,Cylinders,Edispl,Horsepower,Weight,Accelerate,Year]\nTable continents, columns = [*,ContId,Continent]\nTable countries, columns = [*,CountryId,CountryName,Continent]\nTable model_list, columns = [*,ModelId,Maker,Model]\nForeign_keys = [countries.Continent = continents.ContId,car_makers.Country = countries.CountryId,model_list.Maker = car_makers.Id,car_names.Model = model_list.Model,cars_data.Id = car_names.MakeId]"
```

针对这个提示词，我们可以提一个具体的问题：
```
"Q: how many car makers are their in germany?"
```

并且有一个预期答案：
```
"A: SELECT count ( * )  FROM CAR_MAKERS AS T1 JOIN COUNTRIES AS T2 ON T1.Country   =   T2.CountryId WHERE T2.CountryName   =   'germany'"
```

数据集需要采用以下格式：
```
"input": [{"role": "system", "content": "<input prompt>"}, {"role": "user", "content": <user input>}, "ideal": "correct answer"]
```

将它们组合在一起，得到：
```
{"input": [{"role": "system", "content": "TASK: Answer the following question with syntactically correct SQLite SQL. The SQL should be correct and be in context of the previous question-answer pairs.\nTable car_makers, columns = [*,Id,Maker,FullName,Country]\nTable car_names, columns = [*,MakeId,Model,Make]\nTable cars_data, columns = [*,Id,MPG,Cylinders,Edispl,Horsepower,Weight,Accelerate,Year]\nTable continents, columns = [*,ContId,Continent]\nTable countries, columns = [*,CountryId,CountryName,Continent]\nTable model_list, columns = [*,ModelId,Maker,Model]\nForeign_keys = [countries.Continent = continents.ContId,car_makers.Country = countries.CountryId,model_list.Maker = car_makers.Id,car_names.Model = model_list.Model,cars_data.Id = car_names.MakeId]\n"}, {"role": "system", "content": "Q: how many car makers are their in germany"}, "ideal": ["A: SELECT count ( * )  FROM CAR_MAKERS AS T1 JOIN COUNTRIES AS T2 ON T1.Country   =   T2.CountryId WHERE T2.CountryName   =   'germany'"]}
```


加快构建 eval 数据集的一种方法是使用 `GPT-4` 生成合成数据

In [2]:
## Use GPT-4 to generate synthetic data
# Define the system prompt and user input (these should be filled as per the specific use case)
system_prompt = """You are a helpful assistant that can ask questions about a database table and write SQL queries to answer the question.
    A user will pass in a table schema and your job is to return a question answer pairing. The question should relevant to the schema of the table,
    and you can speculate on its contents. You will then have to generate a SQL query to answer the question. Below are some examples of what this should look like.

    Example 1
    ```````````
    User input: Table museum, columns = [*,Museum_ID,Name,Num_of_Staff,Open_Year]\nTable visit, columns = [*,Museum_ID,visitor_ID,Num_of_Ticket,Total_spent]\nTable visitor, columns = [*,ID,Name,Level_of_membership,Age]\nForeign_keys = [visit.visitor_ID = visitor.ID,visit.Museum_ID = museum.Museum_ID]\n
    Assistant Response:
    Q: How many visitors have visited the museum with the most staff?
    A: SELECT count ( * )  FROM VISIT AS T1 JOIN MUSEUM AS T2 ON T1.Museum_ID   =   T2.Museum_ID WHERE T2.Num_of_Staff   =   ( SELECT max ( Num_of_Staff )  FROM MUSEUM ) 
    ```````````

    Example 2
    ```````````
    User input: Table museum, columns = [*,Museum_ID,Name,Num_of_Staff,Open_Year]\nTable visit, columns = [*,Museum_ID,visitor_ID,Num_of_Ticket,Total_spent]\nTable visitor, columns = [*,ID,Name,Level_of_membership,Age]\nForeign_keys = [visit.visitor_ID = visitor.ID,visit.Museum_ID = museum.Museum_ID]\n
    Assistant Response:
    Q: What are the names who have a membership level higher than 4?
    A: SELECT Name   FROM VISITOR AS T1 WHERE T1.Level_of_membership   >   4 
    ```````````

    Example 3
    ```````````
    User input: Table museum, columns = [*,Museum_ID,Name,Num_of_Staff,Open_Year]\nTable visit, columns = [*,Museum_ID,visitor_ID,Num_of_Ticket,Total_spent]\nTable visitor, columns = [*,ID,Name,Level_of_membership,Age]\nForeign_keys = [visit.visitor_ID = visitor.ID,visit.Museum_ID = museum.Museum_ID]\n
    Assistant Response:
    Q: How many tickets of customer id 5?
    A: SELECT count ( * )  FROM VISIT AS T1 JOIN VISITOR AS T2 ON T1.visitor_ID   =   T2.ID WHERE T2.ID   =   5 
    ```````````
    """

user_input = "Table car_makers, columns = [*,Id,Maker,FullName,Country]\nTable car_names, columns = [*,MakeId,Model,Make]\nTable cars_data, columns = [*,Id,MPG,Cylinders,Edispl,Horsepower,Weight,Accelerate,Year]\nTable continents, columns = [*,ContId,Continent]\nTable countries, columns = [*,CountryId,CountryName,Continent]\nTable model_list, columns = [*,ModelId,Maker,Model]\nForeign_keys = [countries.Continent = continents.ContId,car_makers.Country = countries.CountryId,model_list.Maker = car_makers.Id,car_names.Model = model_list.Model,cars_data.Id = car_names.MakeId]"

messages = [{
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_input
    }
]

completion = client.chat.completions.create(
    model="gpt-4-turbo-preview",
    messages=messages,
    temperature=0.7,
    n=5
)

for choice in completion.choices:
    print(choice.message.content + "\n")


Q: What is the average horsepower for cars made in Europe?
A: SELECT AVG(cars_data.Horsepower) FROM cars_data JOIN car_names ON cars_data.Id = car_names.MakeId JOIN model_list ON car_names.Model = model_list.Model JOIN car_makers ON model_list.Maker = car_makers.Id JOIN countries ON car_makers.Country = countries.CountryId JOIN continents ON countries.Continent = continents.ContId WHERE continents.Continent = 'Europe'

Q: What is the average horsepower for cars made in the USA?
A: SELECT AVG(cars_data.Horsepower) FROM cars_data JOIN car_names ON cars_data.Id = car_names.MakeId JOIN car_makers ON car_names.MakeId = car_makers.Id JOIN countries ON car_makers.Country = countries.CountryId WHERE countries.CountryName = 'USA'

Q: What is the average horsepower for cars produced in countries from the continent with the id '3'?
A: SELECT AVG(cars_data.Horsepower) FROM cars_data JOIN car_names ON cars_data.Id = car_names.MakeId JOIN model_list ON car_names.Model = model_list.Model JOIN car_mak

有了合成数据后，我们需要将其转换为与 eval 数据集一致的格式。

In [3]:
eval_data = []
input_prompt = "TASK: Answer the following question with syntactically correct SQLite SQL. The SQL should be correct and be in context of the previous question-answer pairs.\nTable car_makers, columns = [*,Id,Maker,FullName,Country]\nTable car_names, columns = [*,MakeId,Model,Make]\nTable cars_data, columns = [*,Id,MPG,Cylinders,Edispl,Horsepower,Weight,Accelerate,Year]\nTable continents, columns = [*,ContId,Continent]\nTable countries, columns = [*,CountryId,CountryName,Continent]\nTable model_list, columns = [*,ModelId,Maker,Model]\nForeign_keys = [countries.Continent = continents.ContId,car_makers.Country = countries.CountryId,model_list.Maker = car_makers.Id,car_names.Model = model_list.Model,cars_data.Id = car_names.MakeId]"

for choice in completion.choices:
    question = choice.message.content.split("Q: ")[1].split("\n")[0]  # Extracting the question
    answer = choice.message.content.split("\nA: ")[1].split("\n")[0]  # Extracting the answer
    eval_data.append({
        "input": [
            {"role": "system", "content": input_prompt},
            {"role": "user", "content": question},
        ],
        "ideal": answer
    })

for item in eval_data:
    print(item)

{'input': [{'role': 'system', 'content': 'TASK: Answer the following question with syntactically correct SQLite SQL. The SQL should be correct and be in context of the previous question-answer pairs.\nTable car_makers, columns = [*,Id,Maker,FullName,Country]\nTable car_names, columns = [*,MakeId,Model,Make]\nTable cars_data, columns = [*,Id,MPG,Cylinders,Edispl,Horsepower,Weight,Accelerate,Year]\nTable continents, columns = [*,ContId,Continent]\nTable countries, columns = [*,CountryId,CountryName,Continent]\nTable model_list, columns = [*,ModelId,Maker,Model]\nForeign_keys = [countries.Continent = continents.ContId,car_makers.Country = countries.CountryId,model_list.Maker = car_makers.Id,car_names.Model = model_list.Model,cars_data.Id = car_names.MakeId]'}, {'role': 'user', 'content': 'What is the average horsepower for cars made in Europe?'}], 'ideal': "SELECT AVG(cars_data.Horsepower) FROM cars_data JOIN car_names ON cars_data.Id = car_names.MakeId JOIN model_list ON car_names.Model 

接下来，我们需要创建 eval 注册项，以便在框架中运行它。

evals 框架需要一个具有以下属性的 `.yaml` 文件：
* `id` - eval 的标识符
* `description` - eval 的简短描述
* `disclaimer` - 关于 eval 的补充说明
* `metrics` - 我们可以选择三种 eval 指标：match、includes、fuzzyMatch

对于我们的 eval，我们将进行如下配置：

In [4]:
"""
spider-sql:
  id: spider-sql.dev.v0
  metrics: [accuracy]
  description: Eval that scores SQL code from 194 examples in the Spider Text-to-SQL test dataset. The problems are selected by taking the first 10 problems for each database that appears in the test set.
    Yu, Tao, et al. \"Spider; A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task.\" Proceedings of the 2018 Conference on Empirical Methods in Natural Language Processing, 2018, https://doi.org/10.18653/v1/d18-1425.
  disclaimer: Problems are solved zero-shot with no prompting other than the schema; performance may improve with training examples, fine tuning, or a different schema format. Evaluation is currently done through model-grading, where SQL code is not actually executed; the model may judge correct SQL to be incorrect, or vice-versa.
spider-sql.dev.v0:
  class: evals.elsuite.modelgraded.classify:ModelBasedClassify
  args:
    samples_jsonl: sql/spider_sql.jsonl
    eval_type: cot_classify
    modelgraded_spec: sql
  """""

'\nspider-sql:\n  id: spider-sql.dev.v0\n  metrics: [accuracy]\n  description: Eval that scores SQL code from 194 examples in the Spider Text-to-SQL test dataset. The problems are selected by taking the first 10 problems for each database that appears in the test set.\n    Yu, Tao, et al. "Spider; A Large-Scale Human-Labeled Dataset for Complex and Cross-Domain Semantic Parsing and Text-to-SQL Task." Proceedings of the 2018 Conference on Empirical Methods in Natural Language Processing, 2018, https://doi.org/10.18653/v1/d18-1425.\n  disclaimer: Problems are solved zero-shot with no prompting other than the schema; performance may improve with training examples, fine tuning, or a different schema format. Evaluation is currently done through model-grading, where SQL code is not actually executed; the model may judge correct SQL to be incorrect, or vice-versa.\nspider-sql.dev.v0:\n  class: evals.elsuite.modelgraded.classify:ModelBasedClassify\n  args:\n    samples_jsonl: sql/spider_sql.js

## 运行评估

我们可以使用 `oaieval` CLI 运行这个 eval。设置时，请安装该库：`pip install .`（如果你在本地运行 [OpenAI Evals 库](https://github.com/openai/evals)），或者如果你运行的是已有的 eval，则使用 `pip install oaieval`。

然后，使用 CLI 运行 eval：`oaieval gpt-3.5-turbo spider-sql`

该命令需要一个模型名称和一个 eval 集名称。注意，我们提供了两个命令行界面（CLI）：`oaieval` 用于运行单个 eval，`oaievalset` 用于运行一组 evals。有效的 eval 名称在 `evals/registry/evals` 下的 YAML 文件中指定，其对应的实现位于 `evals/elsuite` 中。

In [5]:
!pip install evals --quiet

`oaieval` CLI 可以接受各种标志来修改默认行为。你可以运行 `oaieval --help` 查看完整的 CLI 选项列表。 

运行该命令后，你会看到控制台打印出的最终准确率报告，以及一个指向包含完整报告的临时文件的路径。

`oaieval` 会在 `evals/registry/evals` 目录中查找 `spider-sql` eval 的 YAML 文件，其格式遵循上面第 4 个单元格中的规定。eval 数据集的路径在 eval YAML 文件的 args: 参数下指定为 `samples_jsonl: sql/spider_sql.jsonl`，文件内容为 JSONL 格式（如上面第 3 步中生成的那样）。

运行该命令后，你会看到控制台打印出的最终准确率报告，以及一个指向包含完整报告的临时文件的路径。

In [6]:
!oaieval gpt-3.5-turbo spider-sql --max_samples 25

[2024-03-26 19:44:39,836] [registry.py:257] Loading registry from /Users/shyamal/.virtualenvs/openai/lib/python3.11/site-packages/evals/registry/evals
[2024-03-26 19:44:43,623] [registry.py:257] Loading registry from /Users/shyamal/.evals/evals
[2024-03-26 19:44:43,635] [oaieval.py:189] Run started: 240327024443FACXGMKA
[2024-03-26 19:44:43,663] [registry.py:257] Loading registry from /Users/shyamal/.virtualenvs/openai/lib/python3.11/site-packages/evals/registry/modelgraded
[2024-03-26 19:44:43,851] [registry.py:257] Loading registry from /Users/shyamal/.evals/modelgraded
[2024-03-26 19:44:43,853] [data.py:90] Fetching /Users/shyamal/.virtualenvs/openai/lib/python3.11/site-packages/evals/registry/data/sql/spider_sql.jsonl
[2024-03-26 19:44:43,878] [eval.py:36] Evaluating 25 samples
[2024-03-26 19:44:43,952] [eval.py:144] Running in threaded mode with 10 threads!
  0%|                                                    | 0/25 [00:00<?, ?it/s][2024-03-26 19:44:44,810] [_client.py:1026] H

`oaievalset` 需要一个模型名称和一个 eval 集名称，有效选项在 `evals/registry/eval_sets` 下的 YAML 文件中指定。

### 查看 eval 日志

eval 日志位于 `/tmp/evallogs`，每次评估运行都会创建不同的日志文件。 

In [7]:
log_name = '240327024443FACXGMKA_gpt-3.5-turbo_spider-sql.jsonl' # "EDIT THIS" - copy from above
events = f"/tmp/evallogs/{log_name}"
display(pd.read_json(events, lines=True).head(5))

,spec,final_report,run_id,event_id,sample_id,type,data,created_by,created_at
0,"{'completion_fns': ['gpt-3.5-turbo'], 'eval_name': 'spider-sql.dev.v0', 'base_eval': 'spider-sql', 'split': 'dev', 'run_config': {'completion_fns': ['gpt-3.5-turbo'], 'eval_spec': {'cls': 'evals.elsuite.modelgraded.classify:ModelBasedClassify', 'registry_path': '/Users/shyamal/.virtualenvs/openai/lib/python3.11/site-packages/evals/registry', 'args': {'samples_jsonl': 'sql/spider_sql.jsonl', 'eval_type': 'cot_classify', 'modelgraded_spec': 'sql'}, 'key': 'spider-sql.dev.v0', 'group': 'sql'}, 'seed': 20220722, 'max_samples': 25, 'command': '/Users/shyamal/.virtualenvs/openai/bin/oaieval gpt-3.5-turbo spider-sql --max_samples 25', 'initial_settings': {'visible': False}}, 'created_by': '', 'run_id': '240327024443FACXGMKA', 'created_at': '2024-03-27 02:44:43.626043'}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
1,NaN,"{'counts/Correct': 20, 'counts/Incorrect': 5, 'score': 0.8}",NaN,NaN,NaN,NaN,NaN,NaN,NaT
2,NaN,NaN,240327024443FACXGMKA,0.0,spider-sql.dev.88,sampling,"{'prompt': [{'content': 'Answer the following question with syntactically correct SQLite SQL. Be creative but the SQL must be correct. Use only the following tables and columns: Table: players. Columns: player_id (number), first_name (text), last_name (text), hand (text), birth_date (time), country_code (text) Table: matches. Columns: best_of (number), draw_size (number), loser_age (number), loser_entry (text), loser_hand (text), loser_ht (number), loser_id (number), loser_ioc (text), loser_name (text), loser_rank (number), loser_rank_points (number), loser_seed (number), match_num (number), minutes (number), round (text), score (text), surface (text), tourney_date (time), tourney_id (text), tourney_level (text), tourney_name (text), winner_age (number), winner_entry (text), winner_hand (text), winner_ht (number), winner_id (number), winner_ioc (text), winner_name (text), winner_rank (number), winner_rank_points (number), winner_seed (number), year (number) Table: rankings. Columns: ranking_date (time), ranking (number), player_id (number), ranking_points (number), tours (number) Question: Find the average rank of winners in all matches. ', 'role': 'system'}], 'sampled': ['SELECT AVG(winner_rank) AS average_rank_of_winners FROM matches;']}",,2024-03-27 02:44:44.821110+00:00
3,NaN,NaN,240327024443FACXGMKA,1.0,spider-sql.dev.82,sampling,"{'prompt': [{'content': 'Answer the following question with syntactically correct SQLite SQL. Be creative but the SQL must be correct. Use only the following tables and columns: Table: players. Columns: player_id (number), first_name (text), last_name (text), hand (text), birth_date (time), country_code (text) Table: matches. Columns: best_of (number), draw_size (number), loser_age (number), loser_entry (text), loser_hand (text), loser_ht (number), loser_id (number), loser_ioc (text), loser_name (text), loser_rank (number), loser_rank_points (number), loser_seed (number), match_num (number), minutes (number), round (text), score (text), surface (text), tourney_date (time), tourney_id (text), tourney_level (text), tourney_name (text), winner_age (number), winner_entry (text), winner_hand (text), winner_ht (number), winner_id (number), winner_ioc (text), winner_name (text), winner_rank (number), winner_rank_points (number), winner_seed (number), year (number) Table: rankings. Columns: ranking_date (time), ranking (number), player_id (number), ranking_points (number), tours (number) Question: Find the total number of matches. ', 'role': 'system'}], 'sampled': ['SELECT COUNT(*) AS total_matches FROM matches;']}",,2024-03-27 02:44:44.831848+00:00
4,NaN,NaN,240327024443FACXGMKA,2.0,spider-sql.dev.25,sampling,"{'prompt': [{'content': 'Answer the following question with syntactically correct SQLite SQL. Be creative but the SQL must be correct. Use only the following tables and columns: Table: continents. Columns: ContId (number), Continent (text) Table: countries. Columns: Count

In [8]:
# processing the log events generated by oaieval

with open(events, "r") as f:
    events_df = pd.read_json(f, lines=True)

该文件包含评估的结构化日志。第一个条目提供了评估的详细规格，包括补全函数、评估名称、运行配置、创建者姓名、运行 ID 和创建时间戳。

In [9]:
display(events_df.iloc[0].spec)

{'completion_fns': ['gpt-3.5-turbo'],
 'eval_name': 'spider-sql.dev.v0',
 'base_eval': 'spider-sql',
 'split': 'dev',
 'run_config': {'completion_fns': ['gpt-3.5-turbo'],
  'eval_spec': {'cls': 'evals.elsuite.modelgraded.classify:ModelBasedClassify',
   'registry_path': '/Users/shyamal/.virtualenvs/openai/lib/python3.11/site-packages/evals/registry',
   'args': {'samples_jsonl': 'sql/spider_sql.jsonl',
    'eval_type': 'cot_classify',
    'modelgraded_spec': 'sql'},
   'key': 'spider-sql.dev.v0',
   'group': 'sql'},
  'seed': 20220722,
  'max_samples': 25,
  'command': '/Users/shyamal/.virtualenvs/openai/bin/oaieval gpt-3.5-turbo spider-sql --max_samples 25',
  'initial_settings': {'visible': False}},
 'created_by': '',
 'run_id': '240327024443FACXGMKA',
 'created_at': '2024-03-27 02:44:43.626043'}

我们再来看看提供评估最终报告的条目。

In [10]:
display(events_df.dropna(subset=['final_report']).iloc[0]['final_report'])

{'counts/Correct': 20, 'counts/Incorrect': 5, 'score': 0.8}

我们还可以查看各个评估事件，这些事件提供了具体的样本（`sample_id`）、结果、事件类型和元数据。

In [11]:
pd.set_option('display.max_colwidth', None)  # None means no truncation
display(events_df.iloc[2][['run_id', 'event_id', 'sample_id', 'type', 'data', 'created_at']])

run_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [12]:
# Inspect samples
for i, row in events_df[events_df['type'] == 'sampling'].head(5).iterrows():
    data = pd.json_normalize(row['data'])
    print(f"Prompt: {data['prompt'].iloc[0]}")
    print(f"Sampled: {data['sampled'].iloc[0]}")
    print("-" * 10)

Prompt: [{'content': 'Answer the following question with syntactically correct SQLite SQL. Be creative but the SQL must be correct.\nUse only the following tables and columns:\nTable: players. Columns: player_id (number), first_name (text), last_name (text), hand (text), birth_date (time), country_code (text)\nTable: matches. Columns: best_of (number), draw_size (number), loser_age (number), loser_entry (text), loser_hand (text), loser_ht (number), loser_id (number), loser_ioc (text), loser_name (text), loser_rank (number), loser_rank_points (number), loser_seed (number), match_num (number), minutes (number), round (text), score (text), surface (text), tourney_date (time), tourney_id (text), tourney_level (text), tourney_name (text), winner_age (number), winner_entry (text), winner_hand (text), winner_ht (number), winner_id (number), winner_ioc (text), winner_name (text), winner_rank (number), winner_rank_points (number), winner_seed (number), year (number)\nTable: rankings. Columns: r

我们来查看失败的案例，了解哪些测试没有通过。

In [13]:
def pretty_print_text(prompt):
    # Define markers for the sections
    markers = {
        "question": "[Question]:",
        "expert": "[Expert]:",
        "submission": "[Submission]:",
        "end": "[END DATA]"
    }
    
    # Function to extract text between markers
    def extract_text(start_marker, end_marker):
        start = prompt.find(start_marker) + len(start_marker)
        end = prompt.find(end_marker)
        text = prompt[start:end].strip()
        if start_marker == markers["question"]:
            text = text.split("\n\nQuestion:")[-1].strip() if "\n\nQuestion:" in text else text
        elif start_marker == markers["submission"]:
            text = text.replace("```sql", "").replace("```", "").strip()
        return text
    
    # Extracting text for each section
    question_text = extract_text(markers["question"], markers["expert"])
    expert_text = extract_text(markers["expert"], markers["submission"])
    submission_text = extract_text(markers["submission"], markers["end"])
    
    # HTML color codes and formatting
    colors = {
        "question": '<span style="color: #0000FF;">QUESTION:<br>', 
        "expert": '<span style="color: #008000;">EXPECTED:<br>',  
        "submission": '<span style="color: #FFA500;">SUBMISSION:<br>' 
    }
    color_end = '</span>'
    
    # Display each section with color
    from IPython.display import display, HTML
    display(HTML(f"{colors['question']}{question_text}{color_end}"))
    display(HTML(f"{colors['expert']}{expert_text}{color_end}"))
    display(HTML(f"{colors['submission']}{submission_text}{color_end}"))

In [14]:
# Inspect metrics where choice is made and print only the prompt, result, and expected result if the choice is incorrect
for i, row in events_df[events_df['type'] == 'metrics'].iterrows():
    if row['data']['choice'] == 'Incorrect':
        # Get the previous row's data, which contains the prompt and the expected result
        prev_row = events_df.iloc[i-1]
        prompt = prev_row['data']['prompt'][0]['content'] if 'prompt' in prev_row['data'] and len(prev_row['data']['prompt']) > 0 else "Prompt not available"
        expected_result = prev_row['data'].get('ideal', 'Expected result not provided')
        
        # Current row's data will be the actual result
        result = row['data'].get('result', 'Actual result not provided')
        
        pretty_print_text(prompt)
        print("-" * 40)

----------------------------------------


----------------------------------------


----------------------------------------


----------------------------------------


----------------------------------------


查看部分失败案例后，我们发现：
* 第二个错误答案对 'Templates' 表进行了不必要的连接。我们的 eval 准确识别出了这一点，并将其标记为错误。 
* 另外一些答案存在细微的语法差异，导致答案被标记为错误。
  * 在这种情况下，值得探讨的是：我们应该继续迭代提示词以确保某些风格选择，还是应该修改评估套件以容纳这种差异。
  * 这类失败表明，可能需要使用模型评分的 evals 来确保结果评分的准确性

# 结论

构建有效的 evals 是基于 LLM 的应用开发周期中的核心环节。OpenAI Evals 框架开箱即提供了构建 evals 的核心结构，让你能够为各种用例快速创建新的测试。在本指南中，我们逐步演示了如何创建 eval、运行它并分析结果。

本指南中展示的示例代表了 evals 的一个简单用例。在你继续探索这个框架的过程中，我们建议你尝试为实际生产用例创建更复杂的模型评分 evals。祝评估顺利！